# Interpolation of Fields

This tutorial explores interpolation techniques in HCIPy for resampling fields between different grids. We'll cover linear and nearest-neighbor interpolation, working with both regular and unstructured grids, and practical applications in optical simulations.



In [ ]:
from hcipy import *
from hcipy.interpolation import make_linear_interpolator, make_nearest_interpolator
import numpy as np
import matplotlib.pyplot as plt

## Why Interpolation Matters

In optical simulations, we often need to evaluate fields on grids that differ from where they were originally computed. Common scenarios include:

* **Resampling**: Converting between different grid resolutions
* **Coordinate transforms**: Evaluating fields on rotated or distorted grids
* **Multi-scale simulations**: Combining fields from different optical planes
* **Detector simulation**: Sampling continuous fields onto pixelated detectors

HCIPy provides flexible interpolation tools that maintain field structure and handle both regular (separated) and unstructured grids.

## Basic Resampling

Let's start with simple linear interpolation on a regular grid. We'll create a field on a coarse grid and interpolate it onto a finer grid.

In [ ]:
# Create a coarse grid and a fine grid
coarse_grid = make_pupil_grid(32, 2.0)  # 32x32 grid
fine_grid = make_pupil_grid(128, 2.0)   # 128x128 grid covering same area

print(f"Coarse grid: {coarse_grid.dims} points, spacing {coarse_grid.delta}")
print(f"Fine grid: {fine_grid.dims} points, spacing {fine_grid.delta}")

# Create a field on the coarse grid (a simple Gaussian)
x, y = coarse_grid.x, coarse_grid.y
coarse_field = Field(np.exp(-(x**2 + y**2) / 0.5), coarse_grid)

# Create a linear interpolator
interpolator = make_linear_interpolator(coarse_field)

# Interpolate onto the fine grid
fine_field = interpolator(fine_grid)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

imshow_field(coarse_field, ax=axes[0])
axes[0].set_title(f'Original (coarse)\n{coarse_grid.dims}')

imshow_field(fine_field, ax=axes[1])
axes[1].set_title(f'Interpolated (fine)\n{fine_grid.dims}')

# Show the difference
# Create reference field directly on fine grid for comparison
x_fine, y_fine = fine_grid.x, fine_grid.y
reference_field = Field(np.exp(-(x_fine**2 + y_fine**2) / 0.5), fine_grid)
imshow_field(fine_field - reference_field, cmap='RdBu', ax=axes[2])
axes[2].set_title('Interpolation Error')

plt.tight_layout()
plt.show()

# Calculate error statistics
error = np.abs(fine_field - reference_field)
print(f"\nInterpolation error statistics:")
print(f"  Max error: {error.max():.6f}")
print(f"  Mean error: {error.mean():.6f}")

## Linear vs Nearest-Neighbor

Different interpolation methods have different trade-offs. Let's compare linear and nearest-neighbor interpolation.

In [ ]:
# Create a field with sharp features
grid_64 = make_pupil_grid(64, 2.0)
x, y = grid_64.x, grid_64.y

# Create a circular aperture (sharp edge)
sharp_field = Field(make_circular_aperture(0.8)(grid_64), grid_64)

# Create target grid with different sampling
target_grid = make_pupil_grid(100, 2.0)

# Create interpolators
linear_interp = make_linear_interpolator(sharp_field)
nearest_interp = make_nearest_interpolator(sharp_field)

# Interpolate
linear_result = linear_interp(target_grid)
nearest_result = nearest_interp(target_grid)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

imshow_field(sharp_field, ax=axes[0], cmap='gray')
axes[0].set_title('Original')

imshow_field(linear_result, ax=axes[1], cmap='gray')
axes[1].set_title('Linear Interpolation\n(Smooth, blurs edges)')

imshow_field(nearest_result, ax=axes[2], cmap='gray')
axes[2].set_title('Nearest Neighbor\n(Preserves edges, blocky)')

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Linear: smooth transitions but blurs sharp edges")
print("- Nearest: preserves sharp edges but creates blocky artifacts")

## Handling Boundaries

What happens when we try to interpolate at points outside the original grid? HCIPy allows us to specify fill values for points outside the domain. Note that true extrapolation is not supported for linear interpolation - points outside the domain will use the fill value.

In [ ]:
# Create original field on a small grid
small_grid = make_pupil_grid(50, 1.0)  # Grid spanning [-0.5, 0.5]
x, y = small_grid.x, small_grid.y
original_field = Field(np.exp(-(x**2 + y**2) / 0.1), small_grid)

# Create a larger target grid that extends beyond the original
large_grid = make_pupil_grid(100, 3.0)  # Grid spanning [-1.5, 1.5]

# Different fill values
interpolator_nan = make_linear_interpolator(original_field, fill_value=np.nan)
interpolator_zero = make_linear_interpolator(original_field, fill_value=0.0)
interpolator_const = make_linear_interpolator(original_field, fill_value=0.5)

# Interpolate
result_nan = interpolator_nan(large_grid)
result_zero = interpolator_zero(large_grid)
result_const = interpolator_const(large_grid)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

imshow_field(original_field, ax=axes[0,0])
axes[0,0].set_title('Original Grid\n(span [-0.5, 0.5])')

imshow_field(result_nan, ax=axes[0,1])
axes[0,1].set_title('fill_value=np.nan\n(NaN outside domain)')

imshow_field(result_zero, ax=axes[1,0])
axes[1,0].set_title('fill_value=0.0\n(Zero outside domain)')

imshow_field(result_const, ax=axes[1,1])
axes[1,1].set_title('fill_value=0.5\n(Constant outside domain)')

plt.tight_layout()
plt.show()

print("The white regions in the top-right plot are NaN values.")
print("Note: Linear interpolation does not support extrapolation.")
print("Points outside the domain use the specified fill value.")

## Working with Scattered Data

HCIPy can interpolate from unstructured grids (like scattered data points) using different algorithms optimized for this case.

In [ ]:
from hcipy.field import UnstructuredCoords, CartesianGrid

# Create unstructured grid (scattered points)
np.random.seed(42)
n_points = 500
x_scattered = np.random.uniform(-1, 1, n_points)
y_scattered = np.random.uniform(-1, 1, n_points)

coords = UnstructuredCoords((x_scattered, y_scattered))
unstructured_grid = CartesianGrid(coords)

# Create field values on unstructured grid (some pattern)
values = np.sin(5 * x_scattered) * np.cos(5 * y_scattered)
unstructured_field = Field(values, unstructured_grid)

# Create a regular target grid
regular_grid = make_pupil_grid(100, 2.0)

# Interpolate from unstructured to regular grid
interpolator_unstruct = make_linear_interpolator(unstructured_field)
regular_field = interpolator_unstruct(regular_grid)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot scattered data
sc = axes[0].scatter(x_scattered, y_scattered, c=values, cmap='viridis', s=10)
axes[0].set_aspect('equal')
axes[0].set_title(f'Unstructured Data\n({n_points} scattered points)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
plt.colorbar(sc, ax=axes[0])

# Plot interpolated field
imshow_field(regular_field, ax=axes[1])
axes[1].set_title('Interpolated to Regular Grid')

plt.tight_layout()
plt.show()

print(f"Unstructured grid size: {unstructured_grid.size} points")
print(f"Regular grid size: {regular_grid.size} points")

## Summary

In this tutorial, we explored interpolation techniques in HCIPy:

1. **Basic Interpolation**: Resampling fields between different grid resolutions using linear interpolation
2. **Method Comparison**: Linear interpolation (smooth, blurs edges) vs nearest-neighbor (preserves edges, blocky)
3. **Boundary Handling**: Using fill values for points outside the original domain (extrapolation not supported)
4. **Unstructured Grids**: Interpolating scattered data onto regular grids